In [1]:
# https://housing.com/in/buy/searches/P1od1w26jrfqap1jlS1Uw5eoY1

In [2]:
from bs4 import BeautifulSoup

with open('./Search -- P1od1w26jrfqap1jlS1Uw5eoY1.html', 'r', encoding='utf-8') as f:
    html = f.read()

soup = BeautifulSoup(html, 'html.parser')

In [3]:
pip show natsort

Name: natsort
Version: 8.4.0
Summary: Simple yet flexible natural sorting in Python.
Home-page: https://github.com/SethMMorton/natsort
Author: Seth M. Morton
Author-email: drtuba78@gmail.com
License: MIT
Location: /home/jain/anaconda3/envs/myenv/lib/python3.12/site-packages
Requires: 
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
from natsort import natsorted

In [5]:

# Find all the div elements that satisfy the given condition
div_elements = soup.select('div.T_configContainer[data-q=property-config] > div.T_configurationStyle')

# Create an empty list to store the text values
text_values = []

# Iterate over the div elements and extract the text values
for div in div_elements:
    text_values.append(div.get_text(strip=True))

# Print the list of text values
print(text_values)

['50 sq.yd Plot', '100 sq.yd Plot', '150 sq.yd Plot', '200 sq.yd Plot', '1 BHK Flat', '2 BHK Flat', '450 sq.ft Plot', '675 sq.ft Plot', '900 sq.ft Plot', '1251 sq.ft Plot', '1323 sq.ft Plot', '1611 sq.ft Plot', '1800 sq.ft Plot', '2205 sq.ft Plot', '4680 sq.ft Plot', '1 BHK Flat', '2 BHK Flat', '3 BHK Flat', '1 BHK Flat', '3 BHK Flat', '1 BHK Flat', '3 BHK Flat', '1 BHK Flat', '2 BHK Flat', '1 BHK Flat', '2 BHK Flat', '1 BHK Flat', '2 BHK Flat', '1 BHK Flat', '2 BHK Flat', '3 BHK Flat', '50 sq.yd Plot', '55 sq.yd Plot', '58 sq.yd Plot', '60 sq.yd Plot', '70 sq.yd Plot', '83 sq.yd Plot', '90 sq.yd Plot', '92 sq.yd Plot', '100 sq.yd Plot', '105 sq.yd Plot', '128 sq.yd Plot', '133 sq.yd Plot', '135 sq.yd Plot', '143 sq.yd Plot', '145 sq.yd Plot', '145 sq.yd Plot', '150 sq.yd Plot', '200 sq.yd Plot', '1 BHK Flat', '2 BHK Flat', '1 BHK Flat', '2 BHK Flat', '3 BHK Flat', 'Studio Apartment', '2 BHK Flat', '50 sq.yd Plot', '100 sq.yd Plot', '450 sq.ft Plot', '693 sq.ft Plot', '900 sq.ft Plot',

In [6]:
property_types = list(natsorted(list(set(text_values))))

In [7]:
pip show pandas

Name: pandas
Version: 2.3.3
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License

 Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
 All rights reserved.

 Copyright (c) 2011-2023, Open source contributors.

 Redistribution and use in source and binary forms, with or without
 modification, are permitted provided that the following conditions are met:

 * Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

 * Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

 * Neither the name of the copyright holder nor the names of its
   contribut

In [8]:
import pandas as pd

In [9]:
dict_for_df = {k:[] for k in property_types}
dict_for_df['property_title'] = []
dict_for_df['avg_price'] = []
dict_for_df['possession'] =[]
dict_for_df['seller_label'] = []
dict_for_df['seller_subtitle'] = []
dict_for_df['project'] = []
dict_for_df['single_price'] = []
dict_for_df['builtup_area'] = []

In [10]:
df = pd.DataFrame(dict_for_df)

In [11]:
df

,1 BHK Flat,2 BHK Flat,3 BHK Flat,50 sq.yd Plot,55 sq.yd Plot,58 sq.yd Plot,60 sq.yd Plot,70 sq.yd Plot,83 sq.yd Plot,90 sq.yd Plot,...,4680 sq.ft Plot,Studio Apartment,property_title,avg_price,possession,seller_label,seller_subtitle,project,single_price,builtup_area


In [12]:
import re

In [13]:
# Iterate over div.infoTopContainer to process the inner contents of these div in a for loop
for div in soup.select('div.infoTopContainer'):
    # Process the contents of each div.infoTopContainer
    #print(div)
    
    my_row = {}

    # from this div , fetch div class="title-style"
    title_div = div.select_one('div.title-style')
    if title_div:
        my_row['project'] = title_div.text
        # print(title_div.text)
    else:
        my_row['project'] = "NA"

    
    # from the div, fetch h2 class="subtitle-style" or h2 class="title-style"
    subtitle_div = div.select_one('h2.subtitle-style') or div.select_one('h2.title-style')
    if subtitle_div:
        my_row['property_title'] = subtitle_div.text.strip()
        # print(subtitle_div.text.strip())
    else:
        my_row['property_title'] = "NA"

    
    pricing_dict = {}
    # from this div, fetch all of the div.T_configContainer[data-q=property-config]
    config_containers = div.select('div.T_configContainer[data-q=property-config]')
    for config_container in config_containers:
        lines = config_container.text.splitlines()
        pricing_dict[lines[2].strip()] = lines[4].strip()
    
    my_row.update(pricing_dict)
    # print(pricing_dict)
    
    
    
    # from this div, fetch div.T_configurationStyle[data-q=avg-price]
    avg_price_div = div.select_one('div.T_configurationStyle[data-q=avg-price]')
    if avg_price_div:
        avg_price = avg_price_div.text
    else:
        # from this div, fetch div.property-detail[data-q=property-detail]
        property_detail_div = div.select_one('div.property-detail[data-q=property-detail]')
        if property_detail_div:
            avg_price = property_detail_div.text.split("•")[0].strip()
            # print(property_detail_div.text)
        else:
            avg_price = "NA"
        
    my_row['avg_price'] = avg_price.split(":")[1].strip()
    



    # from this div, fetch div.T_singlePriceStyle[data-q=price]
    price_div = div.select_one('div.T_singlePriceStyle[data-q=price]')
    if price_div:
        my_row["single_price"] = re.sub(r'\s', '', price_div.text)
    else: 
        my_row["single_price"] = "NA"
    
    # from this div, fetch first child of div.T_configContainer[data-q=builtup-area]
    builtup_area_div = div.select_one('div.T_configContainer[data-q=builtup-area]')
    if builtup_area_div:
        my_row['builtup_area'] = builtup_area_div.text.splitlines()[2].strip()
    else:
        my_row['builtup_area'] = "NA"    
    
    print(my_row)
    dict_for_df.update(my_row)
    
    for key, value in dict_for_df.items():
        if isinstance(value, list) and len(value) == 0:
            dict_for_df[key] = 'NA'
    
    df = pd.concat([df, pd.DataFrame([dict_for_df])], ignore_index=True)

{'project': 'RBSM Colony Part 1', 'property_title': 'Residential Plots  in New Palam Vihar Phase 1, Sector 110', '50 sq.yd Plot': '₹13 L', '100 sq.yd Plot': '₹26 L', '150 sq.yd Plot': '₹39 L', '200 sq.yd Plot': '₹52 L', 'avg_price': '₹2.89k/sq.yd', 'single_price': 'NA', 'builtup_area': 'NA'}
{'project': 'DLF Chisholm Apartments', 'property_title': '1, 2 BHK Flats  in Sector 25, DLF Phase 2', '1 BHK Flat': '₹11.25 L - 20.25 L', '2 BHK Flat': '₹24.75 L - 42.75 L', 'avg_price': '₹4.5 K/sq.ft', 'single_price': 'NA', 'builtup_area': 'NA'}
{'project': 'SDH Rose City', 'property_title': 'Residential Plots  in Rajoria Nagar', '450 sq.ft Plot': '₹3.5 L', '675 sq.ft Plot': '₹5.24 L', '900 sq.ft Plot': '₹6.99 L', '1251 sq.ft Plot': '₹9.72 L', '1323 sq.ft Plot': '₹10.28 L', '1611 sq.ft Plot': '₹12.52 L', '1800 sq.ft Plot': '₹13.99 L', '2205 sq.ft Plot': '₹17.13 L', '4680 sq.ft Plot': '₹36.36 L', 'avg_price': '₹777.0/sq.ft', 'single_price': 'NA', 'builtup_area': 'NA'}
{'project': 'Adore Prosperity 

In [14]:
df

,1 BHK Flat,2 BHK Flat,3 BHK Flat,50 sq.yd Plot,55 sq.yd Plot,58 sq.yd Plot,60 sq.yd Plot,70 sq.yd Plot,83 sq.yd Plot,90 sq.yd Plot,...,4680 sq.ft Plot,Studio Apartment,property_title,avg_price,possession,seller_label,seller_subtitle,project,single_price,builtup_area
0,NA,NA,NA,₹13 L,NA,NA,NA,NA,NA,NA,...,NA,NA,"Residential Plots in New Palam Vihar Phase 1,...",₹2.89k/sq.yd,NA,NA,NA,RBSM Colony Part 1,NA,NA
1,₹11.25 L - 20.25 L,₹24.75 L - 42.75 L,NA,₹13 L,NA,NA,NA,NA,NA,NA,...,NA,NA,"1, 2 BHK Flats in Sector 25, DLF Phase 2",₹4.5 K/sq.ft,NA,NA,NA,DLF Chisholm Apartments,NA,NA
2,₹11.25 L - 20.25 L,₹24.75 L - 42.75 L,NA,₹13 L,NA,NA,NA,NA,NA,NA,...,₹36.36 L,NA,Residential Plots in Rajoria Nagar,₹777.0/sq.ft,NA,NA,NA,SDH Rose City,NA,NA
3,₹14.66 L - 14.7 L,₹21.24 L - 21.26 L,₹27.48 L - 27.54 L,₹13 L,NA,NA,NA,NA,NA,NA,...,₹36.36 L,NA,"1, 2, 3 BHK Flats in Sector 35, Sohna",₹3.02 K - ₹4.5 K/sq.ft,NA,NA,NA,Adore Prosperity Homes,NA,NA
4,₹13.84 L - 26.4 L,₹21.24 L - 21.26 L,₹26.29 L - 26.4 L,₹13 L,NA,NA,NA,NA,NA,NA,...,₹36.36 L,NA,"1, 3 BHK Flats in Tech Chand Nagar, Sector 104","Dec, 2025",NA,NA,NA,Mahira Homes 104,NA,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,₹6.97 L - 13.09 L,₹20 L,₹21.56 L,₹7.5 L,₹6.6 L,₹7 L,₹7.2 L,₹8.4 L,₹10 L,₹10.8 L,...,₹36.36 L,₹10 L,Residential Plot in Dhankot,₹1.0k/sq.ft,NA,NA,NA,NA,₹4.5L,450 sq.ft
68,₹6.97 L - 13.09 L,₹20 L,₹21.56 L,₹7.5 L,₹6.6 L,₹7 L,₹7.2 L,₹8.4 L,₹10 L,₹10.8 L,...,₹36.36 L,₹10 L,Residential Plot in Sultanpur,₹8.5k/sq.yd,NA,NA,NA,NA,₹4.25L,50 sq.yd
69,₹6.97 L - 13.09 L,₹20 L,₹21.56 L,₹7.5 L,₹6.6 L,₹7 L,₹7.2 L,₹8.4 L,₹10 L,₹10.8 L,...,₹36.36 L,₹10 L,Residential Plot in Shahid Smarak,₹648.00/sq.ft,NA,NA,NA,NA,₹3.5L,540 sq.ft
70,₹6.97 L - 13.09 L,₹20 L,₹21.56 L,₹7.5 L,₹6.6 L,₹7 L,₹7.2 L,₹8.4 L,₹10 L,₹10.8 L,...,₹36.36 L,₹10 L,"2 BHK Flat in Laxman Vihar, Sector 3A",₹471.00/sq.ft,NA,NA,NA,NA,₹3.3L,700 sq.ft


In [15]:
df.to_csv('1_df.csv', index = False)